# 🏛️ LegalDelta — Indian Legal LLM with Information Gain Reward

**Paper:** [LegalDelta (ICASSP 2026, arXiv 2508.12281)](https://arxiv.org/abs/2508.12281)  
**Core Idea:** Dual-mode input (direct answer vs CoT) + Information Gain reward  
**Our Contribution:** Cross-domain transfer to Indian legal data (ILDC → IPC/BNS)

| Stage | What | GPU? | Runtime |
|-------|------|------|---------|
| 1 | Install + Load Data | ❌ | ~5 min |
| 2 | Generate Dual-Mode Pairs (API) | ❌ | ~1 hr |
| 3 | Load Model + Test IG Reward | ✅ | ~15 min |
| 4 | GRPO Training with LegalDelta Reward | ✅ | ~2 hrs |
| 5 | Evaluation + Ablation Table | ✅ | ~1 hr |

> **Stages 1-2 work WITHOUT GPU.** You can test setup and generate data on CPU.

---

# ══════════════════════════════════════════════
# STAGE 1 — Install Dependencies & Load Data
# ══════════════════════════════════════════════

> **No GPU required.** This stage works on CPU.

## 1.1 Install Dependencies

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes
!pip install -q sentencepiece protobuf huggingface_hub
!pip install -q rouge-score bert-score nltk evaluate
!pip install -q openai pandas numpy tqdm psutil

## 1.2 Clone Repo (Kaggle / Colab / Local)

> ⚙️ **Update `REPO_URL` with your GitHub repo URL.**

In [ ]:
import sys, os, json, re, gc, subprocess
import numpy as np
import pandas as pd
from pathlib import Path

# ╔══════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURE: Your GitHub repo URL             ║
# ╚══════════════════════════════════════════════════╝
REPO_URL  = "https://github.com/highintoxic/RLProject.git"
REPO_NAME = "RLProject"

# Detect environment
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

ENV_NAME = 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')
print(f"🖥️  Environment: {ENV_NAME}")

# Set working directory
if IS_KAGGLE:
    WORK_DIR = f'/kaggle/working/{REPO_NAME}'
    CLONE_DIR = '/kaggle/working'
elif IS_COLAB:
    WORK_DIR = f'/content/{REPO_NAME}'
    CLONE_DIR = '/content'
else:
    WORK_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
    CLONE_DIR = None

# Clone if needed
if not IS_LOCAL:
    if not os.path.exists(os.path.join(WORK_DIR, 'src', '__init__.py')):
        print(f"📥 Cloning repo...")
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], cwd=CLONE_DIR, check=True)
        print("✅ Cloned!")
    else:
        print(f"✅ Repo already at {WORK_DIR}")

PROJECT_ROOT = WORK_DIR
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f"📂 Working dir: {os.getcwd()}")
print(f"📦 src/ found:  {os.path.exists('src/__init__.py')}")

## 1.3 Set API Keys

In [ ]:
from src.config import HF_TOKEN, DEEPSEEK_API_KEY

# Authenticate with HuggingFace
from huggingface_hub import login
if HF_TOKEN:
    login(HF_TOKEN)
    print("✅ HuggingFace authenticated")
else:
    print("⚠️  HF_TOKEN not set — some datasets may be restricted")

print(f"DeepSeek API: {'✅ Found' if DEEPSEEK_API_KEY else '❌ Missing (needed in Stage 2)'}")

## 1.4 Load & Explore ILDC Dataset (No GPU needed!)

In [ ]:
from datasets import load_dataset
from src.config import ILDC_DATASET, NYAYA_DATASET

ildc = load_dataset(ILDC_DATASET, split="train")

print(f"=== ILDC Dataset ===")
print(f"Total cases: {len(ildc)}")
print(f"Columns: {ildc.column_names}")
print(f"\n=== Sample (first 400 chars) ===")
for key, val in ildc[0].items():
    print(f"  {key}: {str(val)[:400]}")

In [ ]:
# Load NyayaAnumana too
try:
    nyaya = load_dataset(NYAYA_DATASET, split="train")
    print(f"\n=== NyayaAnumana ===")
    print(f"Total cases: {len(nyaya)}")
    print(f"Columns: {nyaya.column_names}")
except Exception as e:
    print(f"⚠️ Couldn't load NyayaAnumana: {e}")
    nyaya = None

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  ✅ STAGE 1 CHECKPOINT                   ║
# ║  Deps installed, repo cloned, data       ║
# ║  loaded. NO GPU used yet!                ║
# ╚══════════════════════════════════════════╝
print("✅ Stage 1 complete — datasets loaded, no GPU needed!")

---
# ══════════════════════════════════════════════
# STAGE 2 — Generate Dual-Mode Pairs (API)
# ══════════════════════════════════════════════

> **No GPU.** Uses DeepSeek API. Cost: ~$1.50 for 300 cases.
>
> LegalDelta Stage 1: "Distill from Large Reasoning Model"

In [ ]:
DUAL_MODE_PATH = "dual_mode_data.jsonl"

DUAL_MODE_EXISTS = os.path.exists(DUAL_MODE_PATH)

if DUAL_MODE_EXISTS:
    with open(DUAL_MODE_PATH, 'r', encoding='utf-8') as f:
        existing = len(f.readlines())
    print(f"⏩ CHECKPOINT FOUND: {DUAL_MODE_PATH} ({existing} pairs)")
    print(f"   Skipping generation. Delete file to regenerate.")
else:
    print(f"No dual-mode data found. Will generate 300 pairs.")

## 2.1 Test Single Dual-Mode Pair

In [ ]:
if not DUAL_MODE_EXISTS:
    from src.legaldelta import create_deepseek_client, generate_dual_mode_pair
    
    client = create_deepseek_client()
    
    test_facts = ildc[0].get('facts', ildc[0].get('text', ''))[:1000]
    pair = generate_dual_mode_pair(client, test_facts)
    
    print("=== DIRECT ANSWER (first 400 chars) ===")
    print(pair['direct_answer'][:400])
    print()
    print("=== CoT REASONING (first 400 chars) ===")
    print(pair['cot_reasoning'][:400])
    print()
    print("=== CoT ANSWER (first 400 chars) ===")
    print(pair['cot_answer'][:400])
else:
    print("⏩ Skipped — using existing checkpoint.")

## 2.2 Generate Full Batch (300 Pairs)

In [ ]:
if not DUAL_MODE_EXISTS:
    from src.legaldelta import generate_dual_mode_batch
    
    dual_data = generate_dual_mode_batch(
        ildc_dataset=ildc,
        client=client,
        n_samples=300,
        save_path=DUAL_MODE_PATH,
        save_every=50,
    )
    print(f"\n🎉 Generated {len(dual_data)} dual-mode pairs!")
else:
    print("⏩ Skipped — using existing checkpoint.")

## 2.3 Load & Inspect Dual-Mode Data

In [ ]:
from src.legaldelta import load_dual_mode_data

dual_data = load_dual_mode_data(DUAL_MODE_PATH)

# Stats
print(f"\n=== Dual-Mode Dataset Stats ===")
print(f"  Total pairs: {len(dual_data)}")
direct_lens = [len(d['direct_answer']) for d in dual_data]
cot_lens = [len(d.get('cot_answer', '')) for d in dual_data]
print(f"  Avg direct answer length: {np.mean(direct_lens):.0f} chars")
print(f"  Avg CoT answer length:    {np.mean(cot_lens):.0f} chars")

# Preview one pair
print(f"\n=== Sample Pair #0 ===")
print(f"  Facts:    {dual_data[0]['facts'][:200]}...")
print(f"  Direct:   {dual_data[0]['direct_answer'][:200]}...")
print(f"  CoT:      {dual_data[0]['cot_answer'][:200]}...")

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  ✅ STAGE 2 CHECKPOINT                   ║
# ║  Dual-mode data saved. Still no GPU!     ║
# ╚══════════════════════════════════════════╝
print(f"✅ Stage 2 complete. Data at: {DUAL_MODE_PATH}")

---
# ══════════════════════════════════════════════
# STAGE 3 — Load Model + Test IG Reward
# ══════════════════════════════════════════════

> **GPU Required.** Loads Qwen2.5-1.5B in 4-bit (~1.5 GB VRAM).

In [ ]:
import torch

if not torch.cuda.is_available():
    print("❌ CUDA not available. Enable GPU: Settings → Accelerator → GPU T4.")
    print("   Stages 1-2 are done. Restart with GPU for Stage 3+.")
else:
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3.1 Load Base Model

In [ ]:
from src.model_utils import load_base_model

model, tokenizer = load_base_model()

## 3.2 Test Information Gain on Sample

In [ ]:
from src.legaldelta import compute_information_gain, compute_legaldelta_reward

# Test IG on a sample pair
sample = dual_data[0]

ig_score = compute_information_gain(
    model, tokenizer,
    facts=sample['facts'],
    direct_answer=sample['direct_answer'],
    cot_response=sample.get('cot_answer', ''),
)

print(f"=== Information Gain Test ===")
print(f"  IG score: {ig_score:.4f}")
print(f"  (> 0.1 means CoT meaningfully helps the answer)")

# Full multidimensional reward
reward = compute_legaldelta_reward(
    facts=sample['facts'],
    direct_answer=sample['direct_answer'],
    cot_full_response=sample.get('cot_answer', ''),
    reference=sample.get('reference', ''),
    ig_score=ig_score,
)

print(f"\n=== LegalDelta Multidimensional Reward ===")
print(f"  Total:     {reward['total']:.4f}")
print(f"  IG:        {reward['ig']:.4f} (weight=0.5)")
print(f"  Structure: {reward['structure']:.4f} (weight=0.3)")
print(f"  Domain:    {reward['domain']:.4f} (weight=0.2)")

## 3.3 IG across Multiple Samples (Sanity Check)

In [ ]:
from tqdm import tqdm

ig_scores = []
n_test = min(10, len(dual_data))

print(f"Computing IG for {n_test} samples...")
for i in range(n_test):
    s = dual_data[i]
    ig = compute_information_gain(
        model, tokenizer,
        facts=s['facts'],
        direct_answer=s['direct_answer'],
        cot_response=s.get('cot_answer', ''),
    )
    ig_scores.append(ig)
    print(f"  Sample {i}: IG = {ig:.4f}")

print(f"\n=== IG Distribution ===")
print(f"  Mean:  {np.mean(ig_scores):.4f}")
print(f"  Min:   {np.min(ig_scores):.4f}")
print(f"  Max:   {np.max(ig_scores):.4f}")
print(f"  > 0.1: {sum(1 for s in ig_scores if s > 0.1)}/{n_test} samples")

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  ✅ STAGE 3 CHECKPOINT                   ║
# ║  Model loaded, IG reward verified.        ║
# ╚══════════════════════════════════════════╝
print("✅ Stage 3 complete. IG reward is working!")

---
# ══════════════════════════════════════════════
# STAGE 4 — GRPO Training with LegalDelta Reward
# ══════════════════════════════════════════════

> **GPU Required.** Runtime: ~2 hrs on T4.

In [ ]:
LEGALDELTA_SAVE_DIR = "./legaldelta-indian-1.5b-final"

LD_EXISTS = os.path.exists(LEGALDELTA_SAVE_DIR) and os.path.exists(
    os.path.join(LEGALDELTA_SAVE_DIR, "adapter_config.json")
)

if LD_EXISTS:
    print(f"⏩ CHECKPOINT FOUND: {LEGALDELTA_SAVE_DIR}")
else:
    print("No checkpoint. Will train with LegalDelta reward.")

## 4.1 Apply LoRA

In [ ]:
if not LD_EXISTS:
    from src.model_utils import apply_lora
    
    model, lora_config = apply_lora(model)
    # Expected: ~12M trainable / 1.5B total (~0.8%)
else:
    print("⏩ Skipped.")

## 4.2 Format Dataset for GRPO

In [ ]:
if not LD_EXISTS:
    from datasets import Dataset
    from src.config import INFERENCE_SYSTEM_PROMPT
    
    def format_for_grpo(row):
        return {
            "prompt": (
                f"<|im_start|>system\n"
                f"{INFERENCE_SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n"
                f"Analyze this case:\n\n{row['facts'][:800]}\n"
                f"<|im_end|>\n"
                f"<|im_start|>assistant\n"
            ),
            "direct_answer": row.get("direct_answer", ""),
            "reference": row.get("reference", ""),
        }
    
    grpo_dataset = Dataset.from_list([format_for_grpo(r) for r in dual_data])
    print(f"GRPO dataset: {len(grpo_dataset)} samples")
    print(f"Columns: {grpo_dataset.column_names}")
else:
    print("⏩ Skipped.")

## 4.3 Define LegalDelta Reward Wrapper for GRPO

In [ ]:
if not LD_EXISTS:
    from src.reward_functions import (
        reward_has_legal_citation,
        reward_has_reasoning,
        reward_bilingual,
    )
    from src.legaldelta import compute_information_gain, compute_legaldelta_reward
    
    def legaldelta_reward(completions, **kwargs):
        """Combined LegalDelta reward for GRPO."""
        rewards = []
        for completion in completions:
            # IG reward (lightweight — use empty direct answer as baseline)
            r = compute_legaldelta_reward(
                facts="",
                direct_answer="",
                cot_full_response=completion,
                reference="",
                ig_score=0.3,  # fixed during training for speed
            )
            rewards.append(r['total'])
        return rewards
    
    ALL_REWARD_FUNCS = [
        legaldelta_reward,
        reward_has_legal_citation,
        reward_has_reasoning,
        reward_bilingual,
    ]
    
    print(f"Registered {len(ALL_REWARD_FUNCS)} reward functions")

## 4.4 Train with GRPO

In [ ]:
if not LD_EXISTS:
    from trl import GRPOTrainer, GRPOConfig
    from src.config import (
        GRPO_LEARNING_RATE, GRPO_BATCH_SIZE, GRPO_GRAD_ACCUM_STEPS,
        GRPO_NUM_GENERATIONS, GRPO_MAX_NEW_TOKENS, GRPO_EPOCHS,
    )
    
    grpo_config = GRPOConfig(
        output_dir="./legaldelta-grpo-checkpoints",
        learning_rate=GRPO_LEARNING_RATE,
        per_device_train_batch_size=GRPO_BATCH_SIZE,
        gradient_accumulation_steps=GRPO_GRAD_ACCUM_STEPS,
        num_generations=GRPO_NUM_GENERATIONS,
        max_new_tokens=GRPO_MAX_NEW_TOKENS,
        num_train_epochs=GRPO_EPOCHS,
        fp16=True,
        report_to="none",
        save_steps=100,
    )
    
    trainer = GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        reward_funcs=ALL_REWARD_FUNCS,
        args=grpo_config,
        train_dataset=grpo_dataset,
    )
    
    print("🚀 Starting GRPO training with LegalDelta rewards...")
    trainer.train()
    
    trainer.save_model(LEGALDELTA_SAVE_DIR)
    print(f"\n✅ Training complete! Saved to: {LEGALDELTA_SAVE_DIR}")
else:
    print("⏩ Skipped — using existing checkpoint.")

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  ✅ STAGE 4 CHECKPOINT                   ║
# ║  LegalDelta-trained model saved.         ║
# ╚══════════════════════════════════════════╝

if not LD_EXISTS and 'trainer' in dir():
    del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"✅ Stage 4 complete.")

---
# ══════════════════════════════════════════════
# STAGE 5 — Evaluation + LegalDelta Ablation
# ══════════════════════════════════════════════

> Reproduces LegalDelta Table 2: ablation comparing Direct vs CoT.

## 5.1 Load Models for Comparison

In [ ]:
from src.model_utils import load_base_model, load_finetuned_model, generate_judgment
from src.config import EVAL_N_SAMPLES, ILDC_DATASET

# Reload fresh base model
if 'model' in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

base_model, tokenizer = load_base_model()

# Load trained model
ft_path = LEGALDELTA_SAVE_DIR
if os.path.exists(ft_path):
    finetuned_model, _ = load_finetuned_model(ft_path)
    print(f"Using LegalDelta model from: {ft_path}")
else:
    print("⚠️ No trained model found — running eval on base model only")
    finetuned_model = None

ildc_test = load_dataset(ILDC_DATASET, split="test")
print(f"Test cases: {len(ildc_test)}")

## 5.2 LegalDelta Ablation Table

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
n_eval = min(50, len(ildc_test))

results = {"base_direct": [], "legaldelta_cot": []}

print(f"Evaluating {n_eval} test cases...")
for i in range(n_eval):
    case = ildc_test[i]
    facts = case.get('text', case.get('facts', ''))[:800]
    ref = str(case.get('label', ''))
    
    # Base model (direct)
    base_out = generate_judgment(base_model, tokenizer, facts, max_new_tokens=256)
    base_rouge = scorer.score(ref, base_out)['rougeL'].fmeasure
    results['base_direct'].append(base_rouge)
    
    # LegalDelta model (CoT)
    if finetuned_model:
        ft_out = generate_judgment(finetuned_model, tokenizer, facts, max_new_tokens=512)
        ft_rouge = scorer.score(ref, ft_out)['rougeL'].fmeasure
        results['legaldelta_cot'].append(ft_rouge)
    
    if (i+1) % 10 == 0:
        print(f"  Done {i+1}/{n_eval}")

# Print LegalDelta-style ablation table
print("\n" + "=" * 55)
print("  LegalDelta Ablation — Indian Legal Data")
print("=" * 55)
print(f"{'Mode':<30} {'ROUGE-L':>10} {'Delta':>10}")
print("-" * 55)

base_avg = np.mean(results['base_direct'])
print(f"{'Base (Direct Answer)':<30} {base_avg:>10.4f} {'baseline':>10}")

if results['legaldelta_cot']:
    ld_avg = np.mean(results['legaldelta_cot'])
    delta = ld_avg - base_avg
    print(f"{'+ LegalDelta CoT':<30} {ld_avg:>10.4f} {delta:>+10.4f}")
print("=" * 55)

## 5.3 Side-by-Side Examples

In [ ]:
for idx in [0, 5, 15]:
    case = ildc_test[idx]
    facts = case.get('text', case.get('facts', ''))[:600]
    
    print(f"\n{'='*60}")
    print(f"CASE {idx}")
    print(f"{'='*60}")
    print(f"Facts: {facts[:200]}...\n")
    
    base_out = generate_judgment(base_model, tokenizer, facts, max_new_tokens=256)
    print(f"--- BASE MODEL ---")
    print(base_out[:400])
    
    if finetuned_model:
        ft_out = generate_judgment(finetuned_model, tokenizer, facts, max_new_tokens=512)
        print(f"\n--- LEGALDELTA MODEL ---")
        print(ft_out[:400])

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  🎉 LEGALDELTA PIPELINE COMPLETE                     ║
# ║                                                      ║
# ║  Paper: LegalDelta (ICASSP 2026, arXiv 2508.12281)  ║
# ║  Our contribution:                                   ║
# ║  → Cross-domain: Chinese law → Indian law (ILDC)    ║
# ║  → Model-size:   14B → 1.5B (resource efficient)   ║
# ║  → Bilingual:    Hindi + English output             ║
# ╚══════════════════════════════════════════════════════╝

print("\n🎉 LegalDelta Indian Legal LLM — COMPLETE")
print(f"\nOutputs:")
for p in [DUAL_MODE_PATH, LEGALDELTA_SAVE_DIR]:
    exists = '✅' if os.path.exists(p) else '❌'
    print(f"  {exists} {p}")